# Week 7 — Neural Language Models: Advanced Task

**Task:** Build a mini predictive text application that accepts a sentence fragment, predicts the next word, and displays prediction confidence. Bonus features included: a simple text-based interface, a larger dataset (full CBK report), and multiple predictions per query.

In [1]:
!pip install tensorflow pdfplumber --quiet

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.utils import to_categorical
import numpy as np
import re

print('TensorFlow version:', tf.__version__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 60.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 96.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.2.0 which is incompatible.
TensorFlow version: 2.20.0


## Upload and Process the Full CBK Annual Report (Bonus: Larger Dataset)

In [2]:
from google.colab import files
import pdfplumber

uploaded = files.upload()
pdf_filename = list(uploaded.keys())[0]

text_data = ''
with pdfplumber.open(pdf_filename) as pdf:
    for page in pdf.pages:
        page_text = page.extract_text()
        if page_text:
            text_data += page_text + ' '

text_data = text_data.lower()
text_data = re.sub(r'[^a-z\s.]', ' ', text_data)
text_data = re.sub(r'\s+', ' ', text_data).strip()

sentences = re.split(r'(?<=[.])\s+', text_data)
sentences = [s.strip() for s in sentences if 5 <= len(s.split()) <= 20]

print(f'Full report processed.')
print(f'Total usable sentences: {len(sentences)}')

Saving 1084981846_2025 Annual Report.pdf to 1084981846_2025 Annual Report.pdf
Full report processed.
Total usable sentences: 818


## Build Training Data and Train the Model

Using the full report (Bonus: larger dataset) instead of a small subset, giving the model a richer vocabulary to learn from.

In [3]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(sentences)
vocab_size = len(tokenizer.word_index) + 1

input_sequences = []
for sentence in sentences:
    token_list = tokenizer.texts_to_sequences([sentence])[0]
    for i in range(1, len(token_list)):
        input_sequences.append(token_list[:i+1])

max_seq_len = max(len(seq) for seq in input_sequences)
input_sequences = pad_sequences(input_sequences, maxlen=max_seq_len, padding='pre')

X = input_sequences[:, :-1]
y = to_categorical(input_sequences[:, -1], num_classes=vocab_size)

print(f'Vocabulary size: {vocab_size}')
print(f'Training sequences: {len(input_sequences)}')
print(f'X shape: {X.shape}, y shape: {y.shape}')

Vocabulary size: 1803
Training sequences: 8104
X shape: (8104, 19), y shape: (8104, 1803)


In [4]:
model = Sequential([
    Embedding(vocab_size, 64, input_length=max_seq_len - 1),
    LSTM(128),
    Dense(vocab_size, activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [5]:
history = model.fit(X, y, epochs=60, verbose=1)

print()
print(f'Final training accuracy: {history.history["accuracy"][-1]:.4f}')

Epoch 1/60
254/254 ━━━━━━━━━━━━━━━━━━━━ 15s 45ms/step - accuracy: 0.0521 - loss: 6.5569
Epoch 2/60
254/254 ━━━━━━━━━━━━━━━━━━━━ 9s 35ms/step - accuracy: 0.0537 - loss: 6.1388
Epoch 3/60
254/254 ━━━━━━━━━━━━━━━━━━━━ 10s 37ms/step - accuracy: 0.0684 - loss: 5.9999
Epoch 4/60
254/254 ━━━━━━━━━━━━━━━━━━━━ 10s 38ms/step - accuracy: 0.0954 - loss: 5.7853
Epoch 5/60
254/254 ━━━━━━━━━━━━━━━━━━━━ 9s 37ms/step - accuracy: 0.1141 - loss: 5.5048
Epoch 6/60
254/254 ━━━━━━━━━━━━━━━━━━━━ 9s 33ms/step - accuracy: 0.1467 - loss: 5.2295
Epoch 7/60
254/254 ━━━━━━━━━━━━━━━━━━━━ 10s 38ms/step - accuracy: 0.1674 - loss: 4.9759
Epoch 8/60
254/254 ━━━━━━━━━━━━━━━━━━━━ 10s 38ms/step - accuracy: 0.1897 - loss: 4.7525
Epoch 9/60
254/254 ━━━━━━━━━━━━━━━━━━━━ 8s 32ms/step - accuracy: 0.2069 - loss: 4.5467
Epoch 10/60
254/254 ━━━━━━━━━━━━━━━━━━━━ 10s 39ms/step - accuracy: 0.2214 - loss: 4.3530
Epoch 11/60
254/254 ━━━━━━━━━━━━━━━━━━━━ 10s 39ms/step - accuracy: 0.2305 - loss: 4.1697
Epoch 12/60
254/254 ━━━━━━━━━━━━━━

## The Mini Predictive Text Application

**Features:**
- Accepts a sentence fragment typed by the user
- Predicts the next word
- Displays prediction confidence
- Bonus: shows multiple top predictions, not just one
- Bonus: simple text-based interactive interface

In [6]:
def predict_top_words(seed_text, model, tokenizer, max_seq_len, top_n=3):
    """Return the top_n most likely next words with confidence scores."""
    token_list = tokenizer.texts_to_sequences([seed_text])[0]
    token_list = pad_sequences([token_list], maxlen=max_seq_len - 1, padding='pre')
    predicted_probs = model.predict(token_list, verbose=0)[0]

    top_indices = np.argsort(predicted_probs)[-top_n:][::-1]

    index_to_word = {index: word for word, index in tokenizer.word_index.items()}

    results = []
    for idx in top_indices:
        word = index_to_word.get(idx, '<unknown>')
        confidence = predicted_probs[idx]
        results.append((word, confidence))

    return results

print('=' * 65)
print('  MINI PREDICTIVE TEXT APPLICATION — CBK REPORT')
print('=' * 65)
print()
print('  Type a sentence fragment and see the predicted next word.')
print('  Example fragments:')
print('    "the central bank"')
print('    "inflation declined"')
print('    "the monetary policy committee"')
print()
print('  Type "quit" to stop.')
print('=' * 65)

while True:
    print()
    fragment = input('  YOUR SENTENCE FRAGMENT: ').strip()

    if fragment.lower() in ['quit', 'exit', 'stop', '']:
        print()
        print('  Application closed.')
        break

    predictions = predict_top_words(fragment, model, tokenizer, max_seq_len, top_n=3)

    print()
    print(f'  Fragment: "{fragment}"')
    print(f'  Top 3 predicted next words:')
    for rank, (word, confidence) in enumerate(predictions, 1):
        print(f'    {rank}. "{word}"   (confidence: {confidence:.2%})')
    print()
    print('  ── Try another fragment or type "quit" to stop ──')

  MINI PREDICTIVE TEXT APPLICATION — CBK REPORT

  Type a sentence fragment and see the predicted next word.
  Example fragments:
    "the central bank"
    "inflation declined"
    "the monetary policy committee"

  Type "quit" to stop.


  Fragment: "the central bank"
  Top 3 predicted next words:
    1. "of"   (confidence: 52.06%)
    2. "financing"   (confidence: 29.06%)
    3. "operations"   (confidence: 5.83%)

  ── Try another fragment or type "quit" to stop ──


  Fragment: "inflation declined"
  Top 3 predicted next words:
    1. "outlook"   (confidence: 26.03%)
    2. "to"   (confidence: 14.32%)
    3. "in"   (confidence: 10.73%)

  ── Try another fragment or type "quit" to stop ──

  YOUR SENTENCE FRAGMENT: quit

  Application closed.
